# Exploratory Analysis — lmsys/toxic-chat

This notebook explores the dataset before fine-tuning:
- Label distribution
- Text length distribution
- Sample inspection
- Basic vocabulary analysis

In [ ]:
import sys
sys.path.insert(0, '..')  # make src importable from notebooks/

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

## 1. Load Dataset

In [ ]:
from src.config import load_config
from src.data import load_toxic_chat

cfg = load_config('../configs/base.yaml')
raw = load_toxic_chat(cfg.data)
print(raw)

In [ ]:
# Convert to pandas for easy exploration
df_train = raw['train'].to_pandas()
df_val   = raw['validation'].to_pandas()
df_test  = raw['test'].to_pandas()

print('Train:', len(df_train), '| Val:', len(df_val), '| Test:', len(df_test))
df_train.head()

## 2. Label Distribution

In [ ]:
label_col = cfg.data.label_field

fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharey=False)
for ax, (split_name, df) in zip(axes, [('train', df_train), ('val', df_val), ('test', df_test)]):
    counts = df[label_col].value_counts().sort_index()
    ax.bar(['Non-toxic\n(0)', 'Toxic\n(1)'], counts.values, color=['#4C72B0', '#DD8452'])
    ax.set_title(f'{split_name} split')
    ax.set_ylabel('Count')
    for i, v in enumerate(counts.values):
        ax.text(i, v + 5, str(v), ha='center', fontweight='bold')

fig.suptitle('Label Distribution per Split', fontsize=14, fontweight='bold')
fig.tight_layout()
plt.show()

pct_toxic = df_train[label_col].mean() * 100
print(f'Train: {pct_toxic:.1f}% toxic — class imbalance ratio ≈ 1:{100/pct_toxic - 1:.0f}')

## 3. Text Length Distribution

In [ ]:
text_col = cfg.data.text_field
df_train['char_len'] = df_train[text_col].fillna('').str.len()
df_train['word_len'] = df_train[text_col].fillna('').str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for label_val, label_name, color in [(0, 'Non-toxic', '#4C72B0'), (1, 'Toxic', '#DD8452')]:
    subset = df_train[df_train[label_col] == label_val]
    axes[0].hist(subset['char_len'], bins=50, alpha=0.6, label=label_name, color=color)
    axes[1].hist(subset['word_len'], bins=50, alpha=0.6, label=label_name, color=color)

axes[0].set(title='Character Length', xlabel='Characters', ylabel='Count')
axes[1].set(title='Word Count', xlabel='Words', ylabel='Count')
for ax in axes:
    ax.legend()

fig.suptitle('Text Length Distribution by Label', fontsize=13, fontweight='bold')
fig.tight_layout()
plt.show()

print(df_train.groupby(label_col)[['char_len', 'word_len']].describe().round(1))

## 4. Tokenizer Coverage at max_length=128

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(cfg.model.name)
MAX_LEN = cfg.model.max_length

sample_texts = df_train[text_col].dropna().tolist()[:2000]
token_counts = [len(tokenizer.encode(t, truncation=False)) for t in sample_texts]

pct_within = np.mean([t <= MAX_LEN for t in token_counts]) * 100
print(f'{pct_within:.1f}% of sampled texts fit within max_length={MAX_LEN}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(token_counts, bins=60, color='steelblue', edgecolor='white')
ax.axvline(MAX_LEN, color='red', linestyle='--', label=f'max_length={MAX_LEN}')
ax.set(title='Token count distribution (sample)', xlabel='Tokens', ylabel='Count')
ax.legend()
fig.tight_layout()
plt.show()

## 5. Sample Inspection

In [ ]:
pd.set_option('display.max_colwidth', 200)

print('=== Toxic examples ===')
display(df_train[df_train[label_col] == 1][[text_col, label_col]].sample(5, random_state=0))

print('\n=== Non-toxic examples ===')
display(df_train[df_train[label_col] == 0][[text_col, label_col]].sample(5, random_state=0))

## 6. Column Overview

In [ ]:
print('Available columns:', df_train.columns.tolist())
df_train.dtypes

## 7. Summary

Key findings:
- The dataset is **heavily imbalanced** — toxic examples are a small minority. This motivates tracking precision/recall/F1 rather than accuracy alone.
- Most texts are short enough to fit within `max_length=128` without truncation.
- There is meaningful signal in text length: toxic messages may tend to be shorter or more direct.

Next steps: proceed with fine-tuning in `scripts/run_train.sh`.